# Federated learning example with Fed-BioMed and TanaT

This tutorial shows how Fed-BioMed can be used with TanaT datasets. 

For more information about TanaT library:
- https://github.com/TanaT-Lab/TanaT
- https://tanat-lab.github.io/TanaT/

## 1. Create 3 nodes components and researcher component

Create a `fbm-components` folder with all your nodes for this tutorial, and go to this folder:

```bash
cd fbm-components
```

Create your 3 nodes components:

```bash
fedbiomed component create --component node --path ./center-0 create
fedbiomed component create --component node --path ./center-1 create
fedbiomed component create --component node --path ./center-2 create
```

Create also your researcher component:

```bash
fedbiomed component create --component researcher --path ./my-researcher create
```

## 2. Generate a synthetic dataset with FedPatientJourney

Generate the TanaT dataset that will be used for this demo.

For this, import the following packages:

```python
from tanat_synthea import FedPatientJourney
from tanat.sequence.shortcuts import build_events
```

Generates 3 centres with 60 patients each, by running:

```python
fpj = FedPatientJourney.demo()
fpj.generate()
```

This creates a folder `data/demo/` with three sub-folders `center_0`, `center_1` and `center_2` containing data.

## 3. Create pool events for each center

For each center i, generate the workspace:

```python
from tanat import set_workspace

set_workspace("path-to-your-node-center-i/data")
```

and run this python command:

```python
pool = build_events(
    fpj.center(i).temporal_data(record_types=["observation"]),
    id_column="PATIENT",
    time_column="DATE",
    static_data=fpj.center(i).static_data(),
    store_name="tanat",
)
```

by replacing index i by 0, 1 or 2 to generate tanat data for centers 0, 1 and 2.

This will create some .csv and .arrow tanat dataset files in each node.

## 4. Add custom dataset to each node

For each fedbiomed node, add a custom dataset running the following commands:

```bash
fedbiomed node -p path-to-your-node-center-i dataset add
```

You will see:

```bash
Please select the data type that you're configuring:
	1) csv
	2) default
	3) mednist
	4) images
	5) medical-folder
	6) custom
select:
```

select 6) for custom dataset, and press enter.

Then you will see:

```bash
Name of the database:
```

provide a name of the database, and press enter. You will then see:

```bash
Tags (separate them by comma and no spaces):
```

type:

```bash
tanat
```

and press enter. Then you will see:

```bash
Description:
```

provide a description, and press enter. You will then see:

```bash
Path to the dataset:
```

copy-paste here the path to the dataset of node i, that should be in:

```bash
path-to-your-node-center-i/data/tanat
```

by replacing the index i by 0, 1 or 2 for adding dataset to centers 0, 1 and 2. 

## 5. Start your nodes and your researcher

Use the command below to start your nodes:

```bash
fedbiomed node -p ./center-i start --debug
```

by replacing the index i by 0, 1 or 2 for adding dataset to centers 0, 1 and 2. 

Then, start the researcher with the command:

```bash
fedbiomed researcher -p ./my-researcher start --debug

## 6. Training plan

Create a new training plan.

In [ ]:
from fedbiomed.common.training_plans import TorchTrainingPlan
from fedbiomed.common.datamanager import DataManager
from fedbiomed.common.dataset import CustomDataset
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F

from tanat.sequence.type.event.pool import EventSequencePool
from tanat.criterion import EntityCriterion

class TanaTObsTrainingPlan(TorchTrainingPlan):

    # ------------------------------------------------------------------
    # Model (same MLP as Stage A)
    # ------------------------------------------------------------------
    def init_model(self, model_args: dict):

        in_dim = model_args["in_features"]

        class MLP(nn.Module):
            def __init__(self, in_dim):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(in_dim, 32),
                    nn.ReLU(),
                    nn.Linear(32, 1),
                    nn.Sigmoid(),
                )
            def forward(self, x):
                print('in model forward')
                print(x)
                return self.net(x).squeeze(-1)

        return MLP(in_dim)

    # ------------------------------------------------------------------
    # Optimiser
    # ------------------------------------------------------------------
    def init_optimizer(self, optimizer_args: dict):
        return torch.optim.Adam(
            self.model().parameters(),
            lr=optimizer_args.get("lr", 1e-3),
        )

    # ------------------------------------------------------------------
    # Node-side dependencies
    # ------------------------------------------------------------------
    def init_dependencies(self):
        return [
            "import polars as pl",
            "from tanat.sequence.type.event.pool import EventSequencePool",
            "from fedbiomed.common.dataset import CustomDataset",
            "from tanat.criterion import EntityCriterion"
        ]

    # ------------------------------------------------------------------
    # Create custom dataset
    # ------------------------------------------------------------------
    class TanatDataset(CustomDataset):
        def read(self):
            """Reads the data"""
            TOP_K = 30
            pool = EventSequencePool(store=self.path)

            top_codes = (
                pool.temporal_data()
                .groupby("CODE")
                .size()
                .sort_values(ascending=False)
                .head(TOP_K)
                .index.tolist()
            )

            pool.filter_entities(
                EntityCriterion(query=pl.col("CODE").is_in(top_codes)),
                inplace=True,
            )
            
            pool.cast_features({"CODE": pl.Categorical})
            tensor, _, _ = pool.to_tensor(
                features="CODE",
                bin_size="1D",
                max_bins=90,
                fill_value=0,
                ohe=True,
            )
    
            X = tensor.mean(axis=1)    # (N, M, K) → (N, K)
    
            y_df = pool.apply(
                exprs=(pl.col("GENDER") == "M").cast(pl.Int8).alias("target"),
                is_static=True,
            )
            y = y_df["target"].to_numpy()

            self.X_train = torch.from_numpy(X)
            self.Y_train = torch.from_numpy(y).float()
    
        def __len__(self):
            """Returns the sample size"""
            return len(self.Y_train)
    
        def get_item(self, idx):
            """Gets single sample from the dataset"""
            return self.X_train[idx], self.Y_train[idx]

    # ------------------------------------------------------------------
    # Data loading (runs locally on each node/TanaT pipeline from Stage A)
    # ------------------------------------------------------------------
    def training_data(self):

        dataset = self.TanatDataset()

        return DataManager(
            dataset
        )

    # ------------------------------------------------------------------
    # Loss
    # ------------------------------------------------------------------
    def training_step(self, data, target):
        return F.binary_cross_entropy(self.model().forward(data), target)

## 7. Instantiate and run the experiment

In [21]:
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage

exp = Experiment(
    tags=["tanat"],
    model_args={"in_features": 30},          # K from Stage A
    training_plan_class=TanaTObsTrainingPlan,
    training_args={
        "loader_args": {"batch_size": 16},
        "optimizer_args": {"lr": 1e-3},
        "epochs": 5,
    },
    round_limit=10,
    aggregator=FedAverage(),
)

2026-06-23 14:48:03,549 fedbiomed INFO - Updating training data. This action will update FederatedDataset, and the nodes that will participate to the experiment.

2026-06-23 14:48:03,559 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_90ab8abe-f4e9-4746-b20b-1c9dad023b59

2026-06-23 14:48:03,560 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_9b1b06fd-1f0f-474c-8fc4-68b1ad3193bf

2026-06-23 14:48:03,561 fedbiomed INFO - Node selected for training -> Default Node Alias
Node ID is -> NODE_59bf2749-a925-4baf-b5a4-3d6c17d62b1e

In [22]:
exp.run()

2026-06-23 14:48:03,917 fedbiomed INFO - Sampled nodes in round 0 ['NODE_90ab8abe-f4e9-4746-b20b-1c9dad023b59', 'NODE_9b1b06fd-1f0f-474c-8fc4-68b1ad3193bf', 'NODE_59bf2749-a925-4baf-b5a4-3d6c17d62b1e']

2026-06-23 14:48:03,920 fedbiomed INFO - Sending request 
					 To: NODE_90ab8abe-f4e9-4746-b20b-1c9dad023b59 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-06-23 14:48:03,920 fedbiomed INFO - Sending request 
					 To: NODE_9b1b06fd-1f0f-474c-8fc4-68b1ad3193bf 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-06-23 14:48:03,920 fedbiomed INFO - Sending request 
					 To: NODE_59bf2749-a925-4baf-b5a4-3d6c17d62b1e 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-06-23 14:48:04,023 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_9b1b06fd-1f0f-474c-8fc4-68b1ad3193bf 
					 Node Name: Default Node Alias 
					 Round 1 Epoch: 1 | Iteration: 1/4 (25%) | Samples: 16/64
 					 Loss: 0.692967 
					 ---------

...

2026-06-23 14:48:06,152 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_59bf2749-a925-4baf-b5a4-3d6c17d62b1e 
					 Node Name: Default Node Alias 
					 Round 10 Epoch: 4 | Iteration: 5/5 (100%) | Samples: 65/65
 					 Loss: 0.790987 
					 ---------

2026-06-23 14:48:06,153 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_59bf2749-a925-4baf-b5a4-3d6c17d62b1e 
					 Node Name: Default Node Alias 
					 Round 10 Epoch: 5 | Iteration: 1/5 (20%) | Samples: 16/80
 					 Loss: 0.651005 
					 ---------

2026-06-23 14:48:06,157 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_59bf2749-a925-4baf-b5a4-3d6c17d62b1e 
					 Node Name: Default Node Alias 
					 Round 10 Epoch: 5 | Iteration: 5/5 (100%) | Samples: 65/65
 					 Loss: 0.789326 
					 ---------

2026-06-23 14:48:06,162 fedbiomed INFO - Nodes that successfully reply in round 9 ['NODE_90ab8abe-f4e9-4746-b20b-1c9dad023b59', 'NODE_9b1b06fd-1f0f-474c-8fc4-68b1ad3193bf', 'NODE_59bf2749-a925-4baf-b5a4-3d6c17d62b1e']

10